<h2>0. Download Example Datasets</h2>

In [ ]:
import os
import urllib.request

In [ ]:
os.mkdir("datasets")

In [ ]:
urllib.request.urlretrieve("https://drive.switch.ch/index.php/s/e8jxLGxhv9vVet7/download", "datasets/EDXS.hspy")
urllib.request.urlretrieve("https://drive.switch.ch/index.php/s/vvJzK2LZsyq8jwA/download", "datasets/LL.hspy")

<h2> I. SmoothNMF </h2>

<h3>&emsp;I.1. Internal Absorption Model</h3>

In [1]:
from BlockWiseSmoothNMFModule.config import *
import BlockWiseSmoothNMFModule.BlockWiseSmoothNMFWrapper as bwsnmf
%matplotlib qt6

In [ ]:
data = hs.load("datasets/EDXS.hspy")

In [ ]:
est = bwsnmf.SmoothNMF(
    dataset = data,
    components = 5,
    maxIter = 5000,
    thickness = 2E-5,
    verbose = True,
    xrayDB = XRAY_200KeV,
    splitLinesElements = ["Ca", "Cu", "Fe", "Ga"],
    energyThresholds = [2.5, 5.0, 5.0, 5.0],
    quantificationElements = ["Al", "Ca", "Fe", "Mg", "Si"],
    absorptionElements = ["Al", "Ca", "Fe", "Ga", "Mg", "O", "Si"]
)

In [ ]:
est.fitTransform()

In [ ]:
spectra, loadings, quant = est.getDecompositionResults()
# Use "outputDirectory" argument of getDecompositionResults() to save the results.

In [ ]:
spectra.plot()

In [ ]:
loadings.plot()

In [ ]:
est.printConcentrationReport()

In [ ]:
R = est.calculateResidual()

In [ ]:
R.plot()

In [ ]:
# Warning : Memork leak, may crash the program. Try running with as much free memory as possible.
bwsnmf.plot_data_model_ROI(data, est.estimator.G, est.estimator.W, est.estimator.H, est.estimator.model.modelElements)

<h3>&emsp;I.2. Preparation For External Absorption Decomposition</h3>

<h4>&emsp;&emsp; I.2.1. Compute Density Map</h4>

In [ ]:
D = est.computeDensityMap()

In [ ]:
D_signal = hs.signals.Signal2D(D.reshape(data.data.shape[0], data.data.shape[1]))
D_signal.axes_manager[0].name = 'x'
D_signal.axes_manager[1].name = 'y'
D_signal.axes_manager[0].units = 'pixels'
D_signal.axes_manager[1].units = 'pixels'

In [ ]:
D_signal.plot(cmap = 'gist_rainbow', vmin = 1.25, vmax = 5)
plt.title("Density Map $(g/cm^3)$")

<h4>&emsp;&emsp; I.2.2. Compute Thickness Map From EELS Low-Loss Data</h4>

In [ ]:
LL = hs.load("datasets/LL.hspy")

In [ ]:
LL.align_zero_loss_peak()

In [ ]:
os.makedirs("output", exist_ok = True)

In [ ]:
T = est.computeThicknessMap(LL = LL, D = D, ZLP_threshold = 6.0, outputFilePath = "output/T.inmf")

In [ ]:
T_signal = hs.signals.Signal2D(T.reshape(data.data.shape[0], data.data.shape[1]))
T_signal.axes_manager[0].name = 'x'
T_signal.axes_manager[1].name = 'y'
T_signal.axes_manager[0].units = 'pixels'
T_signal.axes_manager[1].units = 'pixels'

In [ ]:
T_signal.plot(cmap = 'afmhot')
plt.title("Thickness Map $(nm)$")

<h4>&emsp;&emsp; I.2.3. Compute Absorption Correction Matrix</h4>

In [ ]:
A = est.computeAbsorptionCorrectionMatrix(
    thicknessMapFilePath = "output/T.inmf",
    outputDirectory = "output",
    useOpenMP = True
    )

<h3>&emsp;I.2. External Absorption Model</h3>

In [ ]:
from BlockWiseSmoothNMFModule.config import *
import BlockWiseSmoothNMFModule.BlockWiseSmoothNMFWrapper as bwsnmf
%matplotlib qt6

In [ ]:
data = hs.load("datasets/EDXS.hspy")

In [ ]:
est = bwsnmf.SmoothNMF(
    dataset = data,
    components = 5,
    maxIter = 5000,
    thickness = 2E-5,
    verbose = True,
    absorptionModelType = "EXTERNAL",
    xrayDB = XRAY_200KeV,
    absorptionMatrixFilePath = "output/A.onmf",
    splitLinesElements = ["Ca", "Cu", "Fe", "Ga"],
    energyThresholds = [2.5, 5.0, 5.0, 5.0],
    quantificationElements = ["Al", "Ca", "Fe", "Mg", "Si"],
    absorptionElements = ["Al", "Ca", "Fe", "Ga", "Mg", "O", "Si"]
)

In [ ]:
est.fitTransform()

In [ ]:
spectra, loadings, quant = est.getDecompositionResults()
# Use "outputDirectory" argument of getDecompositionResults() to save the results. Make sure to not overwrite the previous results.

In [ ]:
spectra.plot()

In [ ]:
loadings.plot()

In [ ]:
est.printConcentrationReport()

In [ ]:
R = est.calculateResidual()

In [ ]:
R.plot()

In [ ]:
# Warning : Memork leak, may crash the program. Try running with as much free memory as possible.
bwsnmf.plot_data_model_ROI(data, est.estimator.G, est.estimator.W, est.estimator.H, est.estimator.model.modelElements)

<h2> II. BlockWiseSmoothNMF </h2>

<h3>&emsp;II.1. Determine Optimal Partitioning Scheme</h3>

In [ ]:
from BlockWiseSmoothNMFModule.config import *
import BlockWiseSmoothNMFModule.PartitionTool as part
%matplotlib qt6

In [ ]:
data = hs.load("datasets/EDXS.hspy", lazy = True)

In [ ]:
blocksize = (62, 60)

In [ ]:
D = part.PartitionedEDXSDataset(dataset = data, blocksize = blocksize)

In [ ]:
D.plotBlocks()

In [ ]:
D.blockWiseSVD()

In [ ]:
D.plotExplainedVarianceRatios()

<h3>&emsp;II.2. Internal Absorption Model</h3>

In [ ]:
from BlockWiseSmoothNMFModule.config import *
import BlockWiseSmoothNMFModule.BlockWiseSmoothNMFWrapper as bwnmf
%matplotlib qt6

In [ ]:
os.makedirs("output", exist_ok = True)

In [ ]:
data = hs.load("datasets/EDXS.hspy", lazy = True)

In [ ]:
blocksize = (62, 60)

In [ ]:
componentsVector = np.array([1, 2, 2, 2,
                             1, 2, 3, 2,
                             2, 2, 2, 2,
                             2, 2, 1, 2])

In [ ]:
est = bwnmf.BlockWiseSmoothNMF(
    dataset = data,
    workingDirectory = "output",
    blockShape = blocksize,
    nClusters = 4,
    componentsVector = componentsVector,
    thickness = 2E-5,
    xrayDB = XRAY_200KeV,
    splitLinesElements = ["Ca", "Cu", "Fe", "Ga"],
    energyThresholds = [2.5, 5.0, 5.0, 5.0],
    quantificationElements = ["Al", "Ca", "Fe", "Mg", "Si"],
    absorptionElements = ["Al", "Ca", "Fe", "Ga", "Mg", "O", "Si"]
)

In [ ]:
est.fitTransform(
    spectralClassificationMethod = "REFINED_VCA",
    spatialComputationApproach = "BLOCKWISE",
    spatialComputationAlgorithm = "SVD",
    maxIter = 5000,
    tol = 1E-7,
    verbose = True,
    precomputedW = True,
    writeWBlocks = False
)

In [ ]:
spectra, loadings, quant = est.getDecompositionResults()
# Decomposition results are saved automatically in the "output" directory.

In [ ]:
spectra.plot()

In [ ]:
loadings.plot()

In [ ]:
R = est.calculateResiduals()

In [ ]:
R.plot()

In [ ]:
est.printConcentrationReport()

<h3>&emsp;II.3. Preparation For External Absorption Decomposition</h3>

<h4>&emsp;&emsp; II.3.1. Compute Density Map For Each Block</h4>

In [ ]:
D = est.computeDensityMaps()

In [ ]:
D.shape

In [ ]:
D_signal = hs.signals.Signal2D(D.reshape(data.data.shape[0], data.data.shape[1]))
D_signal.axes_manager[0].name = 'x'
D_signal.axes_manager[1].name = 'y'
D_signal.axes_manager[0].units = 'pixels'
D_signal.axes_manager[1].units = 'pixels'

In [ ]:
D_signal.plot(cmap = 'gist_rainbow', vmin = 1.25, vmax = 5)
plt.title("Density Map $(g/cm^3)$")

<h4>&emsp;&emsp; II.3.2. Compute Thickness Map From EELS Low-Loss Data For Each Block</h4>

In [ ]:
LL = hs.load("datasets/LL.hspy", lazy = True)

In [ ]:
LL.data

In [ ]:
est.computeThicknessMaps(LL = LL, D = D, ZLP_threshold = 6.0)

In [ ]:
T = est.readTBlocks()
T_signal = hs.signals.Signal2D(T.reshape(data.data.shape[0], data.data.shape[1]))
T_signal.axes_manager[0].name = 'x'
T_signal.axes_manager[1].name = 'y'
T_signal.axes_manager[0].units = 'pixels'
T_signal.axes_manager[1].units = 'pixels'

In [ ]:
T_signal.plot(cmap = 'afmhot')
plt.title("Thickness Map $(nm)$")

<h4>&emsp;&emsp; II.3.3. Compute Absorption Correction Matrix For Each Block</h4>

In [ ]:
est.computeAbsorptionCorrectionMatrices()

In [ ]:
A = est.readABlocks()

In [ ]:
A.plot()

<h3>&emsp;II.4. External Absorption Model</h3>

In [ ]:
from BlockWiseSmoothNMFModule.config import *
import BlockWiseSmoothNMFModule.BlockWiseSmoothNMFWrapper as bwnmf
%matplotlib qt6

In [ ]:
os.makedirs("output_ext_abs", exist_ok = True)

In [ ]:
data = hs.load("datasets/EDXS.hspy", lazy = True)

In [ ]:
blocksize = (62, 60)

In [ ]:
componentsVector = np.array([1, 2, 2, 2,
                             1, 2, 3, 2,
                             2, 2, 2, 2,
                             2, 2, 1, 2])

In [ ]:
est = bwnmf.BlockWiseSmoothNMF(
    dataset = data,
    workingDirectory = "output_ext_abs",
    blockShape = blocksize,
    nClusters = 4,
    componentsVector = componentsVector,
    thickness = 2E-5,
    xrayDB = XRAY_200KeV,
    absorptionModelType = "EXTERNAL",
    splitLinesElements = ["Ca", "Cu", "Fe", "Ga"],
    energyThresholds = [2.5, 5.0, 5.0, 5.0],
    quantificationElements = ["Al", "Ca", "Fe", "Mg", "Si"],
    absorptionElements = ["Al", "Ca", "Fe", "Ga", "Mg", "O", "Si"]
)

In [ ]:
est.fitTransform(
    spectralClassificationMethod = "REFINED_VCA",
    spatialComputationApproach = "BLOCKWISE",
    spatialComputationAlgorithm = "SVD",
    maxIter = 5000,
    tol = 1E-7,
    verbose = True,
    precomputedW = False,
    writeWBlocks = True
)

In [ ]:
spectra, loadings, quant = est.getDecompositionResults()
# Decomposition results are saved automatically in the "output_ext_abs" directory.

In [ ]:
est.writeAbsorptionCorrectedXBlocks()

In [ ]:
spectra.plot()

In [ ]:
loadings.plot()

In [ ]:
R = est.calculateResiduals()

In [ ]:
R.plot()

In [ ]:
est.printConcentrationReport()